In [15]:

import pennylane as qml
import numpy as np

# ==========================================
# ETAPA 1: Constantes Físicas do Centro NV
# ==========================================
# Trabalharemos com as unidades em MHz para facilitar a leitura
D = 2870.0      # Zero-Field Splitting (MHz)
gamma = 2.8     # Razão giromagnética do elétron (MHz/Gauss)
Bz_teste = 50.0 # Campo magnético externo aplicado (Ex: 50 Gauss)

# ==========================================
# ETAPA 2: Operadores de Spin mapeados em Pauli
# ==========================================
# Qubits lógicos: fio 0 e fio 1
def Sz_operator():
    """ Constrói o operador Sz = (Z0 - Z1) / 2 """
    term1 = 0.5 * (qml.PauliZ(0) @ qml.Identity(1))
    term2 = -0.5 * (qml.Identity(0) @ qml.PauliZ(1))
    return term1 + term2

def Sz2_operator():
    """ Constrói o operador Sz^2 = (I - Z0*Z1) / 2 """
    term1 = 0.5 * (qml.Identity(0) @ qml.Identity(1))
    term2 = -0.5 * (qml.PauliZ(0) @ qml.PauliZ(1))
    return term1 + term2

# ==========================================
# ETAPA 3: Montagem do Hamiltoniano
# ==========================================
def create_nv_hamiltonian(Bz):
    """ Retorna o Hamiltoniano H = D*Sz^2 + gamma*Bz*Sz """
    Sz = Sz_operator()
    Sz2 = Sz2_operator()
    
    # O PennyLane permite criar o Hamiltoniano somando os observáveis multiplicados por escalares
    H = (D * Sz2) + ((gamma * Bz) * Sz)
    return H

# Criando o Hamiltoniano para o nosso campo de teste
hamiltonian = create_nv_hamiltonian(Bz_teste)
print(f"Hamiltoniano criado com sucesso para Bz = {Bz_teste} Gauss!")
print(f"O objeto gerado é do tipo: {type(hamiltonian)}")

Hamiltoniano criado com sucesso para Bz = 50.0 Gauss!
O objeto gerado é do tipo: <class 'pennylane.ops.qubit.hamiltonian.Hamiltonian'>


In [16]:
# ==========================================
# ETAPA 4: O Circuito Quântico (Ansatz)
# ==========================================
# Definindo o dispositivo (simulador quântico) para 2 qubits
dev = qml.device("default.qubit", wires=2)

def ansatz(params):
    """
    Circuito parametrizado para explorar os estados magnéticos.
    Utilizamos rotações RY para amplitude e CNOT para emaranhar os qubits.
    """
    # Camada 1: Rotações independentes
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    
    # Camada de Emaranhamento
    qml.CNOT(wires=[0, 1])
    
    # Camada 2: Rotações de ajuste fino
    qml.RY(params[2], wires=0)
    qml.RY(params[3], wires=1)

# ==========================================
# ETAPA 5: Função de Custo e Loop do VQE
# ==========================================
@qml.qnode(dev)
def cost_function(params):
    # Roda o circuito com os parâmetros atuais
    ansatz(params)
    # Mede o valor esperado do Hamiltoniano criado na Etapa 3
    return qml.expval(hamiltonian)

# Inicializando 4 ângulos aleatórios para as portas RY
np.random.seed(42) # Semente fixa para reprodutibilidade

params = qml.numpy.random.randn(4, requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.1)
max_iterations = 200

print("\n--- Iniciando o treinamento do VQE ---")
for n in range(max_iterations):
    # O otimizador atualiza os parâmetros para minimizar a energia
    params, prev_energy = opt.step_and_cost(cost_function, params)
    
    # Imprimindo o progresso a cada 20 passos
    if (n + 1) % 20 == 0:
        energia_atual = cost_function(params)
        print(f"Iteração {n + 1:3d} | Energia Mínima: {energia_atual:.4f} MHz")

print("\nTreinamento concluído!")
print(f"Parâmetros finais otimizados: {params}")


--- Iniciando o treinamento do VQE ---
Iteração  20 | Energia Mínima: 0.8499 MHz
Iteração  40 | Energia Mínima: 7.7593 MHz
Iteração  60 | Energia Mínima: 0.4778 MHz
Iteração  80 | Energia Mínima: 0.0240 MHz
Iteração 100 | Energia Mínima: 0.0058 MHz
Iteração 120 | Energia Mínima: 0.0024 MHz
Iteração 140 | Energia Mínima: 0.0001 MHz
Iteração 160 | Energia Mínima: 0.0000 MHz
Iteração 180 | Energia Mínima: 0.0000 MHz
Iteração 200 | Energia Mínima: 0.0000 MHz

Treinamento concluído!
Parâmetros finais otimizados: [ 1.27319991 -0.59469441  1.06993777  1.16175291]


In [17]:
# ==========================================
# ETAPA 6: Punição (Deflation) e Estados Excitados
# ==========================================

def penalty_operator(lambda_val):
    """ Constrói o projetor |00><00| em base de Pauli e multiplica por lambda """
    term1 = 0.25 * (qml.Identity(0) @ qml.Identity(1))
    term2 = 0.25 * (qml.PauliZ(0) @ qml.Identity(1))
    term3 = 0.25 * (qml.Identity(0) @ qml.PauliZ(1))
    term4 = 0.25 * (qml.PauliZ(0) @ qml.PauliZ(1))
    
    return lambda_val * (term1 + term2 + term3 + term4)

# Definindo a multa de energia (5000 MHz)
lambda_val = 5000.0

# O Novo Hamiltoniano é a soma do antigo com a punição
H_deflated = hamiltonian + penalty_operator(lambda_val)

# Criamos uma nova função de custo focada neste novo Hamiltoniano
@qml.qnode(dev)
def cost_function_deflated(params):
    ansatz(params)
    return qml.expval(H_deflated)

# Reiniciando os parâmetros aleatoriamente para o novo VQE
np.random.seed(10)
# Forçando o uso do gerador do PennyLane
params_excitado = qml.numpy.random.randn(4, requires_grad=True)
# Usando o AdamOptimizer novamente
opt_deflated = qml.AdamOptimizer(stepsize=0.05)
max_iterations = 250

print("\n--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---")
for n in range(max_iterations):
    params_excitado, prev_energy = opt_deflated.step_and_cost(cost_function_deflated, params_excitado)
    
    if (n + 1) % 25 == 0:
        energia_atual = cost_function_deflated(params_excitado)
        print(f"Iteração {n + 1:3d} | Energia Detectada: {energia_atual:.4f} MHz")

print("\nTreinamento concluído!")


--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---
Iteração  25 | Energia Detectada: 1014.9570 MHz
Iteração  50 | Energia Detectada: 26.1587 MHz
Iteração  75 | Energia Detectada: 1.7196 MHz
Iteração 100 | Energia Detectada: 0.3879 MHz
Iteração 125 | Energia Detectada: 0.0350 MHz
Iteração 150 | Energia Detectada: 0.0025 MHz
Iteração 175 | Energia Detectada: 0.0002 MHz
Iteração 200 | Energia Detectada: 0.0000 MHz
Iteração 225 | Energia Detectada: 0.0000 MHz
Iteração 250 | Energia Detectada: 0.0000 MHz

Treinamento concluído!


In [18]:
# ==========================================
# ETAPA 6: Punição (Deflation) e Penalidade do Fantasma
# ==========================================

def penalty_operator(lambda_val):
    """ Penaliza o estado fundamental |00> """
    term1 = 0.25 * (qml.Identity(0) @ qml.Identity(1))
    term2 = 0.25 * (qml.PauliZ(0) @ qml.Identity(1))
    term3 = 0.25 * (qml.Identity(0) @ qml.PauliZ(1))
    term4 = 0.25 * (qml.PauliZ(0) @ qml.PauliZ(1))
    return lambda_val * (term1 + term2 + term3 + term4)

def ghost_penalty_operator(lambda_val):
    """ Penaliza o estado não-físico |11> """
    term1 = 0.25 * (qml.Identity(0) @ qml.Identity(1))
    term2 = -0.25 * (qml.PauliZ(0) @ qml.Identity(1))  # Sinal invertido
    term3 = -0.25 * (qml.Identity(0) @ qml.PauliZ(1))  # Sinal invertido
    term4 = 0.25 * (qml.PauliZ(0) @ qml.PauliZ(1))
    return lambda_val * (term1 + term2 + term3 + term4)

# Multa de energia (5000 MHz) para ambos os estados proibidos
lambda_val = 5000.0

# O Novo Hamiltoniano bloqueia o |00> e o |11>
H_deflated = hamiltonian + penalty_operator(lambda_val) + ghost_penalty_operator(lambda_val)


# Criamos uma nova função de custo focada neste novo Hamiltoniano
@qml.qnode(dev)
def cost_function_deflated(params):
    ansatz(params)
    return qml.expval(H_deflated)

# Reiniciando os parâmetros aleatoriamente para o novo VQE
np.random.seed(10)
# Forçando o uso do gerador do PennyLane
params_excitado = qml.numpy.random.randn(4, requires_grad=True)
# Usando o AdamOptimizer novamente
opt_deflated = qml.AdamOptimizer(stepsize=0.05)
max_iterations = 250

print("\n--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---")
for n in range(max_iterations):
    params_excitado, prev_energy = opt_deflated.step_and_cost(cost_function_deflated, params_excitado)
    
    if (n + 1) % 25 == 0:
        energia_atual = cost_function_deflated(params_excitado)
        print(f"Iteração {n + 1:3d} | Energia Detectada: {energia_atual:.4f} MHz")

print("\nTreinamento concluído!")


--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---
Iteração  25 | Energia Detectada: 2842.7752 MHz
Iteração  50 | Energia Detectada: 2742.7571 MHz
Iteração  75 | Energia Detectada: 2734.7505 MHz
Iteração 100 | Energia Detectada: 2732.2300 MHz
Iteração 125 | Energia Detectada: 2731.2800 MHz
Iteração 150 | Energia Detectada: 2730.7904 MHz
Iteração 175 | Energia Detectada: 2730.5140 MHz
Iteração 200 | Energia Detectada: 2730.3462 MHz
Iteração 225 | Energia Detectada: 2730.2394 MHz
Iteração 250 | Energia Detectada: 2730.1689 MHz

Treinamento concluído!


In [21]:
# ==========================================
# ETAPA 6: Punição (Deflation) e Penalidade do Fantasma
# ==========================================

def penalty_operator(lambda_val):
    """ Penaliza o estado fundamental |00> """
    term1 = 0.25 * (qml.Identity(0) @ qml.Identity(1))
    term2 = 0.25 * (qml.PauliZ(0) @ qml.Identity(1))
    term3 = 0.25 * (qml.Identity(0) @ qml.PauliZ(1))
    term4 = 0.25 * (qml.PauliZ(0) @ qml.PauliZ(1))
    return lambda_val * (term1 + term2 + term3 + term4)

def ghost_penalty_operator(lambda_val):
    """ Penaliza o estado não-físico |11> """
    term1 = 0.25 * (qml.Identity(0) @ qml.Identity(1))
    term2 = -0.25 * (qml.PauliZ(0) @ qml.Identity(1))  # Sinal invertido
    term3 = -0.25 * (qml.Identity(0) @ qml.PauliZ(1))  # Sinal invertido
    term4 = 0.25 * (qml.PauliZ(0) @ qml.PauliZ(1))
    return lambda_val * (term1 + term2 + term3 + term4)

# Multa de energia (5000 MHz) para ambos os estados proibidos
lambda_val = 5000.0

# O Novo Hamiltoniano bloqueia o |00> e o |11>
H_deflated = hamiltonian + penalty_operator(lambda_val) + ghost_penalty_operator(lambda_val)


# Criamos uma nova função de custo focada neste novo Hamiltoniano
# 1. O QNode quântico faz apenas a medição pura (sem matemática extra)
@qml.qnode(dev)
def measure_deflated_energy(params):
    ansatz(params)
    return qml.expval(H_deflated)

# 2. A Função de Custo clássica inverte o valor para o otimizador maximizar
def cost_function_deflated(params):
    energia_medida = measure_deflated_energy(params)
    return -1.0 * energia_medida

# Reiniciando os parâmetros aleatoriamente para o novo VQE
np.random.seed(10)
# Forçando o uso do gerador do PennyLane
params_excitado = qml.numpy.random.randn(4, requires_grad=True)
# Usando o AdamOptimizer novamente
opt_deflated = qml.AdamOptimizer(stepsize=0.05)
max_iterations = 250

print("\n--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---")
for n in range(max_iterations):
    params_excitado, prev_energy = opt_deflated.step_and_cost(cost_function_deflated, params_excitado)
    
    if (n + 1) % 25 == 0:
        energia_atual = cost_function_deflated(params_excitado)
        print(f"Iteração {n + 1:3d} | Energia Detectada: {energia_atual:.4f} MHz")

print("\nTreinamento concluído!")


--- Iniciando VQE com Deflation (Busca do Estado Excitado) ---
Iteração  25 | Energia Detectada: -4945.1339 MHz
Iteração  50 | Energia Detectada: -4996.0882 MHz
Iteração  75 | Energia Detectada: -4999.8939 MHz
Iteração 100 | Energia Detectada: -4999.9874 MHz
Iteração 125 | Energia Detectada: -4999.9979 MHz
Iteração 150 | Energia Detectada: -4999.9998 MHz
Iteração 175 | Energia Detectada: -5000.0000 MHz
Iteração 200 | Energia Detectada: -5000.0000 MHz
Iteração 225 | Energia Detectada: -5000.0000 MHz
Iteração 250 | Energia Detectada: -5000.0000 MHz

Treinamento concluído!


In [22]:
# 1. O QNode mede o Hamiltoniano ORIGINAL (sem deflation)
@qml.qnode(dev)
def measure_max_energy(params):
    ansatz(params)
    return qml.expval(hamiltonian) # <-- Hamiltoniano puro

# 2. A Função de Custo inverte para achar o teto (+3010 MHz)
def cost_function_max(params):
    return -1.0 * measure_max_energy(params)

# 3. Rodamos o otimizador
np.random.seed(42)
params_max = qml.numpy.random.randn(4, requires_grad=True)
opt_max = qml.AdamOptimizer(stepsize=0.05)

print("\n--- Iniciando VQE (Busca do Estado Máximo) ---")
for n in range(200):
    params_max, prev_energy = opt_max.step_and_cost(cost_function_max, params_max)
    
    if (n + 1) % 20 == 0:
        # Imprimimos o valor real da energia (tirando o sinal de menos)
        energia_real = -1.0 * cost_function_max(params_max)
        print(f"Iteração {n + 1:3d} | Energia Detectada: {energia_real:.4f} MHz")

print("\nTreinamento concluído!")


--- Iniciando VQE (Busca do Estado Máximo) ---
Iteração  20 | Energia Detectada: 2963.0139 MHz
Iteração  40 | Energia Detectada: 3003.7058 MHz
Iteração  60 | Energia Detectada: 3009.2227 MHz
Iteração  80 | Energia Detectada: 3009.6019 MHz
Iteração 100 | Energia Detectada: 3009.8186 MHz
Iteração 120 | Energia Detectada: 3009.9361 MHz
Iteração 140 | Energia Detectada: 3009.9820 MHz
Iteração 160 | Energia Detectada: 3009.9954 MHz
Iteração 180 | Energia Detectada: 3009.9992 MHz
Iteração 200 | Energia Detectada: 3009.9999 MHz

Treinamento concluído!
